## FINETUNE V1 FOR SCIENCE DATA

Run this notebook on Kaggle with both **GPU** and **Internet** enabled. It downloads the public `nhminh107/VietEmbed-RAG-Science` dataset and the `intfloat/multilingual-e5-base` base model, then writes the trained Sentence Transformers model to `/kaggle/working/VietEmbed-RAG-v1-science`.

Every source record is one training triplet: `anchor` → `positive` with `hard_negative` as an explicit negative. The notebook validates all three fields before training and does not drop valid records.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from pathlib import Path

import sentence_transformers
import torch

KAGGLE_WORKING_DIR = Path('/kaggle/working')
DATASET_REPO = 'nhminh107/VietEmbed-RAG-Science'
BASE_MODEL = 'intfloat/multilingual-e5-base'
CHECKPOINT_DIR = KAGGLE_WORKING_DIR / 'vietnamese-embedding-v1.1-science-checkpoints'
FINAL_MODEL_DIR = KAGGLE_WORKING_DIR / 'vietnamese-embedding-v1.1-science'

if not KAGGLE_WORKING_DIR.is_dir():
    raise RuntimeError('This notebook is configured to run on Kaggle.')

os.environ['HF_HOME'] = str(KAGGLE_WORKING_DIR / 'hf-cache')
world_size = int(os.environ.get('WORLD_SIZE', '1'))
if world_size != 1:
    raise RuntimeError('Run this notebook normally with Run All; do not launch it with torchrun.')

gpu_count = torch.cuda.device_count()
if gpu_count == 0:
    raise RuntimeError('No CUDA GPU detected. Enable a GPU accelerator in Kaggle.')

print(f'Sentence Transformers: {sentence_transformers.__version__}')
print(f'Dataset: {DATASET_REPO}')
print(f'Base model: {BASE_MODEL}')
print(f'Final model output: {FINAL_MODEL_DIR}')
for index in range(gpu_count):
    properties = torch.cuda.get_device_properties(index)
    memory_gib = properties.total_memory / 1024**3
    print(f'cuda:{index}: {properties.name} ({memory_gib:.1f} GiB)')

In [ ]:
from datasets import load_dataset

TRIPLET_COLUMNS = ('anchor', 'positive', 'hard_negative')
raw_dataset = load_dataset(DATASET_REPO, split='train')
missing_columns = set(TRIPLET_COLUMNS).difference(raw_dataset.column_names)
if missing_columns:
    raise RuntimeError(
        f'Dataset is missing required triplet columns: {sorted(missing_columns)}'
    )

invalid_rows = raw_dataset.filter(
    lambda example: any(
        not isinstance(example[column], str) or not example[column].strip()
        for column in TRIPLET_COLUMNS
    ),
    desc='Validating triplet fields',
)
if invalid_rows.num_rows:
    raise RuntimeError(
        f'Dataset has {invalid_rows.num_rows:,} invalid triplets; no rows were removed.'
    )

def add_e5_prefixes(example):
    return {
        'anchor': f"query: {example['anchor']}",
        'positive': f"passage: {example['positive']}",
        'hard_negative': f"passage: {example['hard_negative']}",
    }

train_dataset = raw_dataset.map(
    add_e5_prefixes,
    remove_columns=raw_dataset.column_names,
    desc='Adding E5 query/passage prefixes',
)

print(f'Loaded triplets: {train_dataset.num_rows:,}')
print(f'Training columns: {train_dataset.column_names}')
print('Triplet format: anchor=query, positive=passage, hard_negative=passage')

In [ ]:
import math

import transformers

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers

BATCH_SIZE_PER_DEVICE = 24
EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 1

num_records = train_dataset.num_rows
if num_records == 0:
    raise RuntimeError('The dataset contains no valid training triplets.')

samples_per_micro_batch = BATCH_SIZE_PER_DEVICE * world_size
steps_per_epoch = math.ceil(num_records / samples_per_micro_batch)
effective_batch_size = samples_per_micro_batch * GRADIENT_ACCUMULATION_STEPS

print(f'Records per epoch: {num_records:,}')
print(f'Complete data passes: {EPOCHS}')
print(f'Batch size per GPU: {BATCH_SIZE_PER_DEVICE}')
print(f'In-batch negatives per query: {BATCH_SIZE_PER_DEVICE - 1} + 1 explicit hard negative')
print(f'Effective batch size: {effective_batch_size}')
print(f'Optimizer steps per epoch: {steps_per_epoch:,}')

model = SentenceTransformer(BASE_MODEL, device='cuda')
print(f'Model max sequence length: {model.max_seq_length}')
loss = losses.MultipleNegativesRankingLoss(model)

if int(transformers.__version__.split('.', 1)[0]) >= 5:
    warmup_kwargs = {'warmup_steps': 0.1}
else:
    warmup_kwargs = {'warmup_ratio': 0.1}

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_PER_DEVICE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=2e-5,
    **warmup_kwargs,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    dataloader_num_workers=0,
    dataloader_drop_last=False,
    dataloader_pin_memory=True,
    logging_steps=50,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    seed=42,
    data_seed=42,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
)

In [ ]:
trainer.train()
trainer.save_model(str(FINAL_MODEL_DIR))

if not (FINAL_MODEL_DIR / 'modules.json').is_file():
    raise RuntimeError(f'Final model export failed: {FINAL_MODEL_DIR}')

print(f'Final model exported to: {FINAL_MODEL_DIR}')